# Interactive Visual Analytics with Folium
Mark all launch sites, colour-code launch outcomes and measure distances from a launch site to its proximities.

In [1]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [2]:
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


## Task 1 - Mark all launch sites on a map

In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]  
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center')).add_to(site_map)
folium.map.Marker(nasa_coordinate, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
    html='<div style="font-size: 12; color:#d35400;"><b>NASA JSC</b></div>')).add_to(site_map)
for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)
site_map  

## Task 2 - Colour-labelled launch outcomes (green = success, red = failure)

In [4]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
outcome_map = folium.Map(location=nasa_coordinate, zoom_start=4)
marker_cluster = MarkerCluster().add_to(outcome_map)
for _, row in spacex_df.iterrows():
    folium.Marker([row['Lat'], row['Long']],
                  icon=folium.Icon(color='white', icon_color=row['marker_color']),
                  popup=row['Launch Site']).add_to(marker_cluster)
outcome_map   

## Task 3 - Distances from a launch site to its proximities

In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))
site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_coord = (site['Lat'], site['Long'])
proximities = {
    'Coastline': (28.56367, -80.57163),
    'Railway':   (28.57208, -80.58528),
    'Highway':   (28.56420, -80.56800),
    'Titusville (city)': (28.61200, -80.80700),
}
distance_map = folium.Map(location=site_coord, zoom_start=13)
MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
              num_digits=5, prefix='Lat:').add_to(distance_map)
folium.Marker(site_coord, icon=folium.Icon(color='blue'), popup='CCAFS SLC-40').add_to(distance_map)
for name, (lat, lon) in proximities.items():
    d = calculate_distance(site_coord[0], site_coord[1], lat, lon)
    folium.Marker((lat, lon), icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s: %.2f KM</b></div>' % (name, d))).add_to(distance_map)
    folium.PolyLine([site_coord, (lat, lon)], weight=2, color='blue').add_to(distance_map)
    print(f'{name}: {d:.2f} km')
distance_map   

Coastline: 0.51 km
Railway: 1.29 km
Highway: 0.87 km
Titusville (city): 23.13 km


In [6]:
site_map.save('launch_sites_map.html'); outcome_map.save('launch_outcomes_map.html'); distance_map.save('launch_distances_map.html')